# Experiment 1: Synthetic Double-Noise Validation

**Paper**: Engineering Truthiness: A Standard for Pseudo–Ground Truth in Machine Learning Evaluation

**Description**: Controlled simulation demonstrating that calibration can hold while ranking stability fails under structured proxy noise. Validates Theorem 1 and motivates the Silver Risk Index (SRI).

**Requirements**: See `requirements.txt` in the repository root.

**Usage**: Run cells sequentially. All outputs are saved to `outputs/{exp_name}/`.

---

In [ ]:
# =========================
# CONFIGURATION
# =========================
# Public mode: all data is generated synthetically or loaded from public sources.
# To use private data, create config/config_paths_private.py (gitignored).

import os, sys
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Optional: mount Google Drive for private data
    # from google.colab import drive
    # drive.mount('/content/drive')
    pass

# Resolve repo root (works from notebooks/ or repo root)
_nb_dir = Path('.').resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir

# Try private config first, fall back to public defaults
try:
    sys.path.insert(0, str(_repo_root / 'config'))
    from config_paths_private import *  # noqa: F403
    CONFIG_MODE = 'private'
    print('[INFO] Using private path configuration')
except ImportError:
    # Public defaults — everything runs with synthetic data
    REPO_ROOT = _repo_root
    RUN_ID = 'public_run'
    DATADIR = REPO_ROOT / 'data'
    OUTDIR = REPO_ROOT / 'outputs'
    CONFIG_MODE = 'public'
    print('[INFO] Using public path configuration')

Path(OUTDIR).mkdir(parents=True, exist_ok=True)


In [ ]:
import os, json, time, platform, random
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

def utc_stamp():
    import datetime as dt
    return dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")

def make_run_dirs(experiment_name: str):
    run_id = f"{experiment_name}_{utc_stamp()}"
    run_dir = Path(OUTDIR) / experiment_name / run_id
    fig_dir = run_dir / "figures"
    tab_dir = run_dir / "tables"
    met_dir = run_dir / "metrics"
    log_dir = run_dir / "logs"
    for d in [fig_dir, tab_dir, met_dir, log_dir]:
        d.mkdir(parents=True, exist_ok=True)
    return run_id, run_dir, fig_dir, tab_dir, met_dir, log_dir

def save_json(obj: dict, path: Path):
    path.write_text(json.dumps(obj, indent=2))
    return path

print("[OK] Imports/utilities ready.")


In [ ]:
EXP_NAME = "01_synthetic_theory_validation"

# v3-aligned defaults
N_SAMPLES = 20000
N_FEATURES = 20
TRAIN_FRAC = 0.70
SEEDS = list(range(1, 21))         # 20 seeds
NOISE_GRID = [0.0, 0.1, 0.2, 0.3, 0.4]

# Model family used for “multiple models” comparison in rank-flip analysis
C_GRID = [0.05, 0.1, 0.3, 1.0, 3.0, 10.0]

run_id, RUN_DIR, FIG_DIR, TAB_DIR, MET_DIR, LOG_DIR = make_run_dirs(EXP_NAME)

print("[RUN]", RUN_DIR)
print("[PARAM] N_SAMPLES=", N_SAMPLES, "N_FEATURES=", N_FEATURES, "TRAIN_FRAC=", TRAIN_FRAC)
print("[PARAM] SEEDS=", SEEDS)
print("[PARAM] NOISE_GRID=", NOISE_GRID)
print("[PARAM] C_GRID=", C_GRID)


In [ ]:
def gen_synthetic_binary(n: int, d: int, seed: int):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, d))
    w = rng.normal(size=(d,))
    logits = X @ w
    y = (logits > 0).astype(int)
    return X, y

def apply_label_flip(y: np.ndarray, p: float, seed: int):
    rng = np.random.default_rng(seed)
    y2 = y.copy()
    flip = rng.random(len(y2)) < p
    y2[flip] = 1 - y2[flip]
    return y2, float(flip.mean())

def train_test_split_idx(n: int, train_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    return idx[:n_train], idx[n_train:]

print("[OK] Data generation helpers ready.")


In [ ]:
from sklearn.decomposition import PCA

# Use one canonical seed for a quick EDA preview
set_global_seed(SEEDS[0])
X_preview, y_preview = gen_synthetic_binary(N_SAMPLES, N_FEATURES, seed=SEEDS[0])

# PCA to 2D for visualization
pca = PCA(n_components=2, random_state=SEEDS[0])
X2 = pca.fit_transform(X_preview)

idx = np.random.default_rng(SEEDS[0]).choice(len(X2), size=1500, replace=False)

plt.figure(figsize=(6,5))
plt.scatter(X2[idx,0], X2[idx,1], c=y_preview[idx], s=10)
plt.title("Synthetic data PCA(2D) colored by GOLD label (subset)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()

fig0 = FIG_DIR / "fig00_synthetic_pca_scatter_gold.png"
plt.savefig(fig0, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("[SAVED]", fig0)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rows = []
t0 = time.time()

for seed in SEEDS:
    set_global_seed(seed)
    X, y_gold = gen_synthetic_binary(N_SAMPLES, N_FEATURES, seed=seed)
    tr_idx, te_idx = train_test_split_idx(N_SAMPLES, TRAIN_FRAC, seed=seed)

    Xtr, Xte = X[tr_idx], X[te_idx]
    ytr_gold, yte_gold = y_gold[tr_idx], y_gold[te_idx]

    for p in NOISE_GRID:
        y_silver, realized = apply_label_flip(y_gold, p, seed=10_000 + seed + int(1000*p))
        ytr_silver, yte_silver = y_silver[tr_idx], y_silver[te_idx]

        # Train on SILVER train labels (proxy training)
        clf = LogisticRegression(max_iter=800)
        clf.fit(Xtr, ytr_silver)

        # Evaluate on both gold and silver
        yhat_te = clf.predict(Xte)
        acc_gold = accuracy_score(yte_gold, yhat_te)
        acc_silver = accuracy_score(yte_silver, yhat_te)
        gap = abs(acc_silver - acc_gold)

        rows.append({
            "seed": seed,
            "noise_p": p,
            "realized_flip_rate": realized,
            "acc_gold_test": acc_gold,
            "acc_silver_test": acc_silver,
            "abs_gap": gap,
            "n_train": len(tr_idx),
            "n_test": len(te_idx),
        })

df_seed = pd.DataFrame(rows)

elapsed = time.time() - t0
print(f"[TIMER] core loop: {elapsed:.2f}s")

# Display (inline)
df_seed.head(10)


In [ ]:
csv_seed = TAB_DIR / "results_01_synthetic_per_seed.csv"
df_seed.to_csv(csv_seed, index=False)
print("[SAVED]", csv_seed)


In [ ]:
df_summary = (
    df_seed
    .groupby("noise_p", as_index=False)
    .agg(
        realized_flip_rate_mean=("realized_flip_rate", "mean"),
        acc_gold_test_mean=("acc_gold_test", "mean"),
        acc_gold_test_std=("acc_gold_test", "std"),
        acc_silver_test_mean=("acc_silver_test", "mean"),
        acc_silver_test_std=("acc_silver_test", "std"),
        abs_gap_mean=("abs_gap", "mean"),
        abs_gap_std=("abs_gap", "std"),
    )
)

df_summary


In [ ]:
csv_table1 = TAB_DIR / "results_01_synthetic_accuracy_table1.csv"
df_summary.to_csv(csv_table1, index=False)
print("[SAVED]", csv_table1)


In [ ]:
plt.figure(figsize=(8,5))
plt.errorbar(df_summary["noise_p"], df_summary["acc_gold_test_mean"],
             yerr=df_summary["acc_gold_test_std"], marker="o", capsize=3, label="Gold test accuracy")
plt.errorbar(df_summary["noise_p"], df_summary["acc_silver_test_mean"],
             yerr=df_summary["acc_silver_test_std"], marker="x", linestyle="--", capsize=3, label="Silver test accuracy")
plt.xlabel("Noise level p")
plt.ylabel("Accuracy")
plt.title("Gold vs Silver evaluation under increasing label noise (mean±std over 20 seeds)")
plt.grid(True)
plt.legend()
plt.tight_layout()

fig1 = FIG_DIR / "fig01_accuracy_vs_noise_mean_std.png"
plt.savefig(fig1, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("[SAVED]", fig1)


In [ ]:
plt.figure(figsize=(8,5))
plt.errorbar(df_summary["noise_p"], df_summary["abs_gap_mean"],
             yerr=df_summary["abs_gap_std"], marker="o", capsize=3)
plt.xlabel("Noise level p")
plt.ylabel("|Acc_silver - Acc_gold|")
plt.title("Evaluation gap induced by silver labels (mean±std over 20 seeds)")
plt.grid(True)
plt.tight_layout()

fig2 = FIG_DIR / "fig02_abs_gap_vs_noise_mean_std.png"
plt.savefig(fig2, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("[SAVED]", fig2)


In [ ]:
from itertools import combinations

flip_rows = []
t0 = time.time()

for seed in SEEDS:
    set_global_seed(seed)
    X, y_gold = gen_synthetic_binary(N_SAMPLES, N_FEATURES, seed=seed)
    tr_idx, te_idx = train_test_split_idx(N_SAMPLES, TRAIN_FRAC, seed=seed)
    Xtr, Xte = X[tr_idx], X[te_idx]
    yte_gold = y_gold[te_idx]

    for p in NOISE_GRID:
        y_silver, realized = apply_label_flip(y_gold, p, seed=20_000 + seed + int(1000*p))
        ytr_silver = y_silver[tr_idx]
        yte_silver = y_silver[te_idx]

        scores = []
        for C in C_GRID:
            clf = LogisticRegression(max_iter=900, C=C)
            clf.fit(Xtr, ytr_silver)
            yhat = clf.predict(Xte)
            acc_g = accuracy_score(yte_gold, yhat)
            acc_s = accuracy_score(yte_silver, yhat)
            scores.append((C, acc_g, acc_s))

        # pairwise inversions between GOLD ranking and SILVER ranking
        pairs = list(combinations(range(len(C_GRID)), 2))
        inv = 0
        denom = 0
        for i, j in pairs:
            Ci, Cj = C_GRID[i], C_GRID[j]
            gi, gj = scores[i][1], scores[j][1]
            si, sj = scores[i][2], scores[j][2]
            gold_order = np.sign(gi - gj)
            silver_order = np.sign(si - sj)
            if gold_order == 0 or silver_order == 0:
                continue
            denom += 1
            if gold_order != silver_order:
                inv += 1

        flip_rate = inv / denom if denom > 0 else 0.0
        flip_rows.append({
            "seed": seed,
            "noise_p": p,
            "realized_flip_rate": realized,
            "pairwise_inversion_rate": flip_rate,
            "denom_pairs": denom
        })

df_flip_seed = pd.DataFrame(flip_rows)
elapsed = time.time() - t0
print(f"[TIMER] flip loop: {elapsed:.2f}s")

df_flip_seed.head(10)


In [ ]:
csv_flip_seed = TAB_DIR / "results_01_pairwise_flip_rate_per_seed.csv"
df_flip_seed.to_csv(csv_flip_seed, index=False)
print("[SAVED]", csv_flip_seed)


In [ ]:
df_flip = (
    df_flip_seed
    .groupby("noise_p", as_index=False)
    .agg(
        pairwise_inversion_rate_mean=("pairwise_inversion_rate", "mean"),
        pairwise_inversion_rate_std=("pairwise_inversion_rate", "std"),
        realized_flip_rate_mean=("realized_flip_rate", "mean"),
        denom_pairs_mean=("denom_pairs", "mean"),
    )
)

df_flip


In [ ]:
csv_flip = TAB_DIR / "results_01_pairwise_flip_rate_table.csv"
df_flip.to_csv(csv_flip, index=False)
print("[SAVED]", csv_flip)


In [ ]:
plt.figure(figsize=(8,5))
plt.errorbar(df_flip["noise_p"], df_flip["pairwise_inversion_rate_mean"],
             yerr=df_flip["pairwise_inversion_rate_std"], marker="o", capsize=3)
plt.xlabel("Noise level p")
plt.ylabel("Pairwise inversion rate (rank flips)")
plt.title("Ranking flips under SILVER-only evaluation (mean±std over 20 seeds)")
plt.grid(True)
plt.tight_layout()

fig3 = FIG_DIR / "fig03_pairwise_inversion_rate_mean_std.png"
plt.savefig(fig3, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("[SAVED]", fig3)


In [ ]:
manifest = {
    "experiment": EXP_NAME,
    "run_id": run_id,
    "timestamp_utc": run_id.replace(EXP_NAME + "_", ""),
    "config_mode": CONFIG_MODE,
    "params": {
        "N_SAMPLES": N_SAMPLES,
        "N_FEATURES": N_FEATURES,
        "TRAIN_FRAC": TRAIN_FRAC,
        "SEEDS": SEEDS,
        "NOISE_GRID": NOISE_GRID,
        "C_GRID": C_GRID,
    },
    "outputs": {
        "tables": [
            str(TAB_DIR / "results_01_synthetic_per_seed.csv"),
            str(TAB_DIR / "results_01_synthetic_accuracy_table1.csv"),
            str(TAB_DIR / "results_01_pairwise_flip_rate_per_seed.csv"),
            str(TAB_DIR / "results_01_pairwise_flip_rate_table.csv"),
        ],
        "figures": [
            str(FIG_DIR / "fig00_synthetic_pca_scatter_gold.png"),
            str(FIG_DIR / "fig01_accuracy_vs_noise_mean_std.png"),
            str(FIG_DIR / "fig02_abs_gap_vs_noise_mean_std.png"),
            str(FIG_DIR / "fig03_pairwise_inversion_rate_mean_std.png"),
        ],
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
    }
}

manifest_path = RUN_DIR / "run_manifest.json"
save_json(manifest, manifest_path)
print("[SAVED]", manifest_path)
print("[DONE] All artifacts saved under:", RUN_DIR)
